# Assignment 2 - YouTube Data Collection and Cleaning

## 1. Aim

Collect YouTube video metadata and comments for videos discussing AI coding assistants such as GitHub Copilot, Cursor, Claude Code, Codex, AI coding agents, and related developer workflow topics.

The collected data should support:

- NLP/text analysis of comment themes, sentiment, and concerns.
- Network analysis using video-commenter, commenter-video, and co-commenting relationships.

In [ ]:
import json
import re
import string
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import nltk
import pandas as pd

from nltk.corpus import stopwords
from nltk.tokenize import TweetTokenizer

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
project_root_str = str(PROJECT_ROOT.resolve())
if project_root_str not in sys.path:
    sys.path.append(project_root_str)

from src.utils.youtube_client import youtubeClient

nltk.download("stopwords")

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = PROJECT_ROOT / "data/raw/youtube"
PROCESSED_DIR = PROJECT_ROOT / "data/processed/youtube"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

MAX_VIDEOS = 20
MAX_COMMENTS_PER_VIDEO = 200

## 2. Data Collection Function

The collection process follows this structure:

1. Search YouTube for videos matching a query.
2. Pull video-level metadata and statistics.
3. Pull top-level comments for each video.
4. Save one nested JSON file per search query.

In [ ]:
def fetchYoutubeData(
    searchQuery,
    maxVideos=25,
    maxCommentsPerVideo=None,
    outputFile="youtubeDataDump.json",
):
    client = youtubeClient()

    searchResponse = client.search().list(
        q=searchQuery,
        part="snippet",
        type="video",
        order="relevance",
        maxResults=min(maxVideos, 50),
    ).execute()

    videoIds = []
    videoSnippets = {}

    for item in searchResponse.get("items", []):
        videoId = item["id"]["videoId"]
        videoIds.append(videoId)
        videoSnippets[videoId] = item["snippet"]

    videoStats = {}
    if videoIds:
        statsResponse = client.videos().list(
            id=",".join(videoIds),
            part="statistics",
        ).execute()

        for item in statsResponse.get("items", []):
            videoStats[item["id"]] = item["statistics"]

    videos = []

    for searchRank, videoId in enumerate(videoIds, start=1):
        snippet = videoSnippets[videoId]
        stats = videoStats.get(videoId, {})

        video = {
            "sourceQuery": searchQuery,
            "searchRank": searchRank,
            "title": snippet["title"],
            "videoId": videoId,
            "channelTitle": snippet["channelTitle"],
            "publishedAt": snippet["publishedAt"],
            "viewCount": int(stats.get("viewCount", 0)),
            "likeCount": int(stats.get("likeCount", 0)),
            "commentCount": int(stats.get("commentCount", 0)),
            "comments": [],
        }

        try:
            comments_fetched = 0
            next_page_token = None

            while True:
                if maxCommentsPerVideo is not None:
                    remaining = maxCommentsPerVideo - comments_fetched
                    if remaining <= 0:
                        break
                    request_limit = min(100, remaining)
                else:
                    request_limit = 100

                commentResponse = client.commentThreads().list(
                    videoId=videoId,
                    part="snippet",
                    maxResults=request_limit,
                    pageToken=next_page_token,
                    textFormat="plainText",
                ).execute()

                for commentThread in commentResponse.get("items", []):
                    topComment = commentThread["snippet"]["topLevelComment"]["snippet"]

                    video["comments"].append(
                        {
                            "author": topComment["authorDisplayName"],
                            "text": topComment["textDisplay"],
                            "publishedAt": topComment["publishedAt"],
                            "likeCount": topComment.get("likeCount", 0),
                        }
                    )
                    comments_fetched += 1

                    if (
                        maxCommentsPerVideo is not None
                        and comments_fetched >= maxCommentsPerVideo
                    ):
                        break

                if (
                    maxCommentsPerVideo is not None
                    and comments_fetched >= maxCommentsPerVideo
                ):
                    break

                next_page_token = commentResponse.get("nextPageToken")
                if not next_page_token:
                    break

        except Exception as e:
            print(f"Comments disabled or error for video {videoId}: {e}")

        videos.append(video)

    data = {"searchQuery": searchQuery, "videos": videos}

    with open(outputFile, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(videos)} videos to {outputFile}")

## 3. Search Terms

These are starting terms only. After the first run, inspect the video titles and remove terms that bring in irrelevant videos.

In [ ]:
SEARCH_QUERIES = [
    "GitHub Copilot review developers",
    "GitHub Copilot coding assistant",
    "Cursor AI code editor review",
    "Claude Code coding assistant",
    "OpenAI Codex coding agent",
    "AI coding agents software development",
    "AI code assistant programmer productivity",
    "vibe coding developers",
    "AI replacing programmers software developers",
    "AI coding assistant security privacy",
]

for index, searchQuery in enumerate(SEARCH_QUERIES, start=1):
    output_file = RAW_DIR / f"youtube_ai_coding_{index}.json"
    fetchYoutubeData(
        searchQuery,
        maxVideos=MAX_VIDEOS,
        maxCommentsPerVideo=MAX_COMMENTS_PER_VIDEO,
        outputFile=str(output_file),
    )

## 4. Convert Nested JSON into a Comment Table

The raw JSON is nested by video. This converts it into one row per comment, with video metadata attached to each comment.

In [ ]:
videos_parsed = []
seen_video_ids = set()

for json_file in sorted(RAW_DIR.glob("youtube_ai_coding_*.json")):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    for video in data.get("videos", []):
        video_id = video["videoId"]

        if video.get("comments") and video_id not in seen_video_ids:
            videos_parsed.append(video)
            seen_video_ids.add(video_id)

video_comments = []
for video in videos_parsed:
    for comment in video["comments"]:
        video_comments.append({
            "sourceQuery": video.get("sourceQuery"),
            "searchRank": video.get("searchRank"),
            "videoId": video["videoId"],
            "title": video["title"],
            "channelTitle": video["channelTitle"],
            "publishedAt": video["publishedAt"],
            "viewCount": video["viewCount"],
            "likeCount": video["likeCount"],
            "videoCommentCount": video.get("commentCount", 0),
            "commentAuthor": comment["author"],
            "commentText": comment["text"],
            "commentPublishedAt": comment["publishedAt"],
            "commentLikeCount": comment["likeCount"],
        })

comments_df = pd.DataFrame(video_comments)
comments_df.to_csv(PROCESSED_DIR / "YTCommentsRawFlattened.csv", index=False)

print(f"Unique videos with comments: {len(videos_parsed)}")
print(f"Total comments collected: {len(video_comments)}")
comments_df.head()

## 5. Basic Data Checks

In [ ]:
comments_df.shape

comments_df.info()

video_summary = comments_df.groupby([
    "videoId",
    "title",
    "channelTitle",
    "publishedAt",
]).agg(
    collectedCommentCount=("commentText", "count"),
    viewCount=("viewCount", "first"),
    likeCount=("likeCount", "first"),
    videoCommentCount=("videoCommentCount", "first"),
).reset_index()

video_summary.sort_values("collectedCommentCount", ascending=False).head(20)

In [ ]:
comments_df["publishedAt"] = pd.to_datetime(comments_df["publishedAt"], errors="coerce")
comments_df["commentPublishedAt"] = pd.to_datetime(comments_df["commentPublishedAt"], errors="coerce")

print("Video date range:", comments_df["publishedAt"].min(), "to", comments_df["publishedAt"].max())
print("Comment date range:", comments_df["commentPublishedAt"].min(), "to", comments_df["commentPublishedAt"].max())

video_summary["publishedAt"] = pd.to_datetime(video_summary["publishedAt"], errors="coerce")
video_summary["year"] = video_summary["publishedAt"].dt.year

video_summary["year"].value_counts().sort_index().plot(kind="bar", figsize=(8, 5))
plt.title("Selected Videos by Publication Year")
plt.xlabel("Year")
plt.ylabel("Number of Videos")
plt.tight_layout()
plt.show()

## 6. Title Relevance Check

This is a rough first-pass check. The final video list should still be manually reviewed before analysis.

In [ ]:
print("Video titles in the dataset:")
for videoTitle in comments_df["title"].drop_duplicates():
    print(videoTitle)


def processText(text, tokenizer, stemmer, stopwords_list):
    text = str(text).lower()
    lTokens = tokenizer.tokenize(text)
    lTokens = [token.strip() for token in lTokens]
    lStemmedTokens = set([stemmer.stem(tok) for tok in lTokens])

    return [tok for tok in lStemmedTokens if tok not in stopwords_list and not tok.isdigit()]


termFreqCounter = Counter()

commentTokeniser = TweetTokenizer()
lPunct = list(string.punctuation)
lStopwords = stopwords.words("english") + lPunct + ["via"]
commentStemmer = nltk.stem.PorterStemmer()

for videoTitle in comments_df["title"].drop_duplicates():
    lTokens = processText(
        text=videoTitle,
        tokenizer=commentTokeniser,
        stemmer=commentStemmer,
        stopwords_list=lStopwords,
    )
    termFreqCounter.update(lTokens)

top_terms = termFreqCounter.most_common(20)
print("\nMost common terms in video titles:", top_terms)

if top_terms:
    terms, frequencies = zip(*top_terms)
    plt.figure(figsize=(10, 5))
    plt.bar(terms, frequencies)
    plt.title("Top Terms in Video Titles")
    plt.xlabel("Terms")
    plt.ylabel("Frequency")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
include_keywords = [
    "ai",
    "copilot",
    "github copilot",
    "cursor",
    "claude code",
    "codex",
    "coding assistant",
    "code assistant",
    "coding agent",
    "developer",
    "developers",
    "programmer",
    "programmers",
    "software engineer",
    "software development",
    "vibe coding",
    "productivity",
    "security",
    "privacy",
]


def relevance_score(title):
    title_lower = str(title).lower()
    return sum(keyword in title_lower for keyword in include_keywords)


video_summary["relevanceScore"] = video_summary["title"].apply(relevance_score)

top_videos = (
    video_summary.sort_values(["relevanceScore", "collectedCommentCount"], ascending=False)
    [["videoId", "title", "channelTitle", "publishedAt", "collectedCommentCount", "relevanceScore"]]
)

print(top_videos.to_string(index=False))

## 7. Manual Video Selection

After reviewing the title list above, paste the video IDs to keep into `selected_video_ids`. If this list is empty, the notebook keeps all videos that passed the earlier collection step.

In [ ]:
selected_video_ids = [
    # Example:
    # "abc123xyz",
]

if selected_video_ids:
    filtered_comments_df = comments_df[comments_df["videoId"].isin(selected_video_ids)].copy()
else:
    filtered_comments_df = comments_df.copy()

print("Selected videos:", filtered_comments_df["videoId"].nunique())
print("Selected comments:", len(filtered_comments_df))

## 8. Data Cleaning

This stage removes duplicate comments, removes empty comments, converts dates, and adds simple text-cleaning fields for later NLP.

In [ ]:
before_rows = len(filtered_comments_df)

filtered_comments_df = filtered_comments_df.drop_duplicates(
    subset=["videoId", "commentAuthor", "commentText"]
).copy()

after_rows = len(filtered_comments_df)

print(f"Removed duplicate comments: {before_rows - after_rows}")
print(f"Remaining comments: {after_rows}")

In [ ]:
filtered_comments_df = filtered_comments_df[filtered_comments_df["commentText"].notna()].copy()
filtered_comments_df = filtered_comments_df[
    filtered_comments_df["commentText"].astype(str).str.strip() != ""
].copy()

filtered_comments_df["publishedAt"] = pd.to_datetime(
    filtered_comments_df["publishedAt"],
    errors="coerce",
)

filtered_comments_df["commentPublishedAt"] = pd.to_datetime(
    filtered_comments_df["commentPublishedAt"],
    errors="coerce",
)

filtered_comments_df = filtered_comments_df[
    filtered_comments_df["commentPublishedAt"].notna()
].copy()

print(f"Dataset shape after empty-comment/date cleaning: {filtered_comments_df.shape}")
print("Comment date range:", filtered_comments_df["commentPublishedAt"].min(), "to", filtered_comments_df["commentPublishedAt"].max())

In [ ]:
def clean_comment_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


filtered_comments_df["commentTextClean"] = filtered_comments_df["commentText"].apply(clean_comment_text)
filtered_comments_df["commentTextLower"] = filtered_comments_df["commentTextClean"].str.lower()
filtered_comments_df["commentLength"] = filtered_comments_df["commentTextClean"].str.len()

filtered_comments_df = filtered_comments_df[filtered_comments_df["commentLength"] > 0].copy()

filtered_comments_df.head()

In [ ]:
filtered_comments_df.to_csv(PROCESSED_DIR / "YTCommentsCleaned.csv", index=False)

final_video_summary = filtered_comments_df.groupby([
    "videoId",
    "title",
    "channelTitle",
]).agg(
    collectedCommentCount=("commentTextClean", "count"),
    uniqueCommenters=("commentAuthor", "nunique"),
    averageCommentLength=("commentLength", "mean"),
    viewCount=("viewCount", "first"),
    likeCount=("likeCount", "first"),
).reset_index()

final_video_summary.to_csv(PROCESSED_DIR / "YTVideoSummary.csv", index=False)

print("Saved:")
print(PROCESSED_DIR / "YTCommentsCleaned.csv")
print(PROCESSED_DIR / "YTVideoSummary.csv")

print("\nFinal dataset shape:", filtered_comments_df.shape)
print("Final unique videos:", filtered_comments_df["videoId"].nunique())
print("Final unique commenters:", filtered_comments_df["commentAuthor"].nunique())
print("Final total comments:", len(filtered_comments_df))

## 9. Network Edge Tables

These edge tables are used as inputs for the network analysis stage.

- `video_commenter_edges`: connects each video to each commenter.
- `commenter_video_edges`: same relationship in the opposite column order, useful for bipartite graphs.

In [ ]:
video_commenter_edges = (
    filtered_comments_df
    .groupby(["videoId", "commentAuthor"])
    .agg(weight=("commentTextClean", "count"))
    .reset_index()
)

commenter_video_edges = video_commenter_edges.rename(
    columns={"commentAuthor": "source", "videoId": "target"}
)[["source", "target", "weight"]]

video_commenter_edges.to_csv(PROCESSED_DIR / "YTVideoCommenterEdges.csv", index=False)
commenter_video_edges.to_csv(PROCESSED_DIR / "YTCommenterVideoEdges.csv", index=False)

video_commenter_edges.head()